# basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
from openai import OpenAI

BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "qwen2.5:7b"
LLM_OPTIONS = {"reasoning_effort": "none"} if API_KEY == "ollama" else {}
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


In [ ]:
# A special phrase the model can output when it believes the task is complete.
# In later modules this will be replaced by a structured "tool call" signal,
# but for Module 1 (no tools yet) a plain text marker is simplest to reason about.
DONE_MARKER = "TASK_COMPLETE"

# Safety valve: never loop forever. This cap exists purely so a buggy or
# confused model can't run up API costs indefinitely.
MAX_TURNS = 6

# System prompt: tells the model how the loop works so it can cooperate with it.
SYSTEM_PROMPT = f"""You are a helpful assistant operating inside an agent loop.
You will be given a task and can respond over multiple turns to think it through.
When you are fully done and ready to give your final answer, start your message
with the exact marker "{DONE_MARKER}" followed by your final answer.
If you are not yet done, just respond normally and you will be prompted to continue.
"""

# loop

In [5]:
def run_agent_loop(prompt: str) -> str:  
    """
    Runs the minimal agent loop for a single user task.

    The loop structure is the heart of Module 1:
        1. Send the full conversation history to the LLM.
        2. Print what we're about to do (for learning purposes).
        3. Get the LLM's response and print it.
        4. Check if the response contains the DONE_MARKER.
           - If yes: stop and return the final answer.
           - If no: append the response to history, nudge the model to
             continue, and go back to step 1.
    """

    messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ]

    for turn in range(1, MAX_TURNS + 1):
        print(f"\n{'=' * 60}")
        print(f"[STEP {turn}] Sending {len(messages)} messages to the LLM...")
        print(f"{'=' * 60}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            **LLM_OPTIONS,
        )

        assistant_reply = response.choices[0].message.content
        print(f"[STEP {turn}] LLM responded:")
        print(f"---\n{assistant_reply}\n---")

        messages.append({"role": "assistant", "content": assistant_reply})

        if DONE_MARKER in assistant_reply:
            # --- Stopping condition check ---
            final_answer = assistant_reply.split(DONE_MARKER, 1)[1].strip()
            print(f"\n[STEP {turn}] Done marker found. Ending loop.")
            return final_answer

        print(f"[STEP {turn}] No done marker yet. Continuing loop...")
        messages.append({
            "role": "user",
            "content": (
                f"Continue. If you are finished, remember to start your "
                f"reply with '{DONE_MARKER}'."
            ),
        })

    # If we hit MAX_TURNS without a done marker, we stop anyway. This is an
    # important lesson for real agents: always have a hard ceiling on loop
    # length so a confused model can't run forever.
    print("\n[WARNING] Max turns reached without a done marker.")
    return "(Agent did not finish within the turn limit.)"


In [7]:
task = "Explain, in exactly 3 short bullet points, why agent loops need a stopping condition."
print(f"USER TASK: {task}")

result = run_agent_loop(task)


print(f"\n{'#' * 60}")
print("FINAL ANSWER:")
print(result)
print(f"{'#' * 60}")

USER TASK: Explain, in exactly 3 short bullet points, why agent loops need a stopping condition.

[STEP 1] Sending 2 messages to the LLM...
[STEP 1] LLM responded:
---
- Without a stopping condition, an agent loop could run indefinitely, consuming resources and potentially leading to system instability.
- A stopping condition ensures the loop executes a finite number of times, making the process predictable and allowing for timely termination.
- Stopping conditions are crucial for implementing decision-making in the loop, enabling the agent to achieve specific goals and avoid getting stuck in infinite cycles.
---
[STEP 1] No done marker yet. Continuing loop...

[STEP 2] Sending 4 messages to the LLM...
[STEP 2] LLM responded:
---
- Stopping conditions help in managing the computational resources effectively by preventing unnecessary computations and saving energy.
- They allow for the integration of external signals or events, which can trigger the loop to stop based on real-world cond

In [9]:
task = "Explain, what's the result of 1+200 *10."
print(f"USER TASK: {task}")

result = run_agent_loop(task)


print(f"\n{'#' * 60}")
print("FINAL ANSWER:")
print(result)
print(f"{'#' * 60}")

USER TASK: Explain, what's the result of 1+200 *10.

[STEP 1] Sending 2 messages to the LLM...
[STEP 1] LLM responded:
---
To solve the expression \(1 + 200 * 10\), we need to follow the order of operations, often remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division (from left to right), Addition and Subtraction (from left to right)).

1. First, we perform the multiplication: \(200 * 10 = 2000\).
2. Then, we add 1 to the result of the multiplication: \(1 + 2000 = 2001\).

So, the result of \(1 + 200 * 10\) is 2001.
---
[STEP 1] No done marker yet. Continuing loop...

[STEP 2] Sending 4 messages to the LLM...
[STEP 2] LLM responded:
---
TASK_COMPLETE 2001
---

[STEP 2] Done marker found. Ending loop.

############################################################
FINAL ANSWER:
2001
############################################################
